# PART A: CONCEPTUAL FOUNDATION

## What is Data Analysis?

- Data Analysis = process of collecting, cleaning, transforming, and analyzing data
- Goal: extract meaningful insights for decision making

## Data Science Project Plan

- Steps followed in this project:
- 1. Problem Understanding
- 2. Data Collection (CSV, JSON, SQL, API)
- 3. Data Cleaning (missing values, outliers)
- 4. Feature Engineering
- 5. Model Building
- 6. Evaluation

## ML Problem Framing

- Problem Type: Binary Classification
- Target Variable: default_flag (0 = No, 1 = Yes)
- Goal: Predict whether customer will default

## Tensors (NumPy Explanation)

A Tensor is a multi-dimensional array.

Types:
- Scalar (0D)
- Vector (1D)
- Matrix (2D)
- Tensor (3D+)

Used in machine learning to represent structured data.

In [93]:
import numpy as np

# Scalar (0D)
scalar = np.array(5)

# Vector (1D)
vector = np.array([1,2,3])

# Matrix (2D)
matrix = np.array([[1,2],[3,4]])

# Tensor (3D)
tensor = np.array([[[1],[2]], [[3],[4]]])

# WHY TENSORS?
# Used in ML to represent multi-dimensional data
print(tensor.shape)

(2, 2, 1)


# PART B: DATA ACQUISITION

Data is collected from multiple sources:
- CSV → Transaction data
- JSON → Customer details
- SQL → Repayment history
- API → External data

All datasets are merged into one final dataset.

In [94]:
import pandas as pd 
import json           # Used to read JSON files
import sqlite3        # Used to connect SQL database
import requests       # Used to fetch API data
import random 
from datetime import datetime, timedelta

### Loading CSV File

CSV (Comma-Separated Values) is the most common data format.

Why used?
- Easy to store and read
- Contains structured tabular data
- Often used for transaction records

In this project:
transactions.csv contains customer transaction data.

In [95]:
# CSV file contains transactions information

transactions = pd.read_csv(r"C:\Users\tanaa\Downloads\transactions_final.csv")

# Display first 5 rows
print("transactions Data (CSV):")
display(transactions.head())

transactions Data (CSV):


,customer_id,transaction_count,spending_ratio,join_date
0,1,348,0.246,2019-01-23
1,2,218,0.877,2019-06-15
2,3,416,0.187,2019-01-14
3,4,247,0.153,2023-05-09
4,5,306,0.110,2020-03-10


### Loading JSON File

JSON (JavaScript Object Notation) is used for semi-structured data.

Why used?
- Flexible structure
- Common in APIs and web data
- Stores nested data

In this project:
customers.json contains customer details like demographics.

In [96]:
# JSON file contains Customers details

Customers = pd.read_json(r"C:\Users\tanaa\Downloads\customers_final.json")

# Display first 5 rows
print("Customers Data (JSON):")
display(Customers.head())

Customers Data (JSON):


,customer_id,age,gender,education_level,region,employment_type,annual_income
0,1,54,Female,Primary,South,Self-Employed,158466
1,2,21,Female,Secondary,West,Self-Employed,139129
2,3,21,Male,Graduate,South,Self-Employed,26663
3,4,42,Female,Post-Graduate,East,Unemployed,155955
4,5,41,Female,Graduate,South,Self-Employed,59359


### Loading SQL Data

SQL databases store structured data in tables.

Why SQL is used?
- Efficient for large datasets
- Supports relationships (joins)
- Used in banking/finance systems

In this project:
repayment_history table contains loan repayment behavior.

In [97]:
import sqlite3
import pandas as pd
import numpy as np

# Step 1: connection create karo
conn = sqlite3.connect(r"C:\Users\tanaa\Downloads\loans_large.db")

# Step 2: data read karo
repayment = pd.read_sql("SELECT * FROM repayment_history", conn)

# Step 3: new column add karo
repayment["repayment_history"] = np.random.choice(
    ["Good", "Average", "Poor"],
    size=len(repayment)
)

# Step 4: output dekho
print("Repayment Data after adding repayment_history:")
repayment.head()

Repayment Data after adding repayment_history:


,customer_id,loan_amount,credit_score,default_flag,repayment_history
0,1,95213,596,0,Good
1,2,67720,306,1,Average
2,3,40113,373,1,Average
3,4,94946,699,1,Poor
4,5,16431,857,1,Good


## API Data Collection

API (Application Programming Interface) is used to fetch external data.

Why API is used:
- Provides real-time data
- Adds external features to dataset
- Improves model accuracy

In this project, API is used to simulate external economic indicators
like inflation rate, interest rate, etc.

In [98]:
import pandas as pd
import numpy as np

# Simulated API data
api_data = pd.DataFrame({
    "customer_id": transactions["customer_id"],
    "inflation_rate": np.random.uniform(4, 8, len(transactions)),
    "interest_rate": np.random.uniform(6, 12, len(transactions))
})

# Loan Purpose column add (random assignment example)
loan_purposes = ["Home Loan", "Education Loan", "Personal Loan", "Business Loan"]

api_data["loan_purpose"] = np.random.choice(loan_purposes, len(transactions))

# Display first 5 rows
print("API Data:")
display(api_data.head())

API Data:


,customer_id,inflation_rate,interest_rate,loan_purpose
0,1,4.308858,10.079462,Personal Loan
1,2,4.735687,10.034368,Personal Loan
2,3,7.576694,9.180393,Personal Loan
3,4,6.386953,8.390404,Home Loan
4,5,6.697592,11.000303,Business Loan


### Merging Datasets

All datasets are merged using customer_id.

Why merging?
- Each dataset contains partial information
- Combine all features into one dataset
- Required for machine learning model

In [99]:
# Merge all datasets step by step

df = transactions.merge(Customers, on="customer_id")
df = df.merge(repayment, on="customer_id")
df = df.merge(api_data, on="customer_id")

# Display final dataset
print("Final Merged Dataset:")
display(df.head())

# WHY?
# Create a complete dataset for analysis and modeling

Final Merged Dataset:


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,region,employment_type,annual_income,loan_amount,credit_score,default_flag,repayment_history,inflation_rate,interest_rate,loan_purpose
0,1,348,0.246,2019-01-23,54,Female,Primary,South,Self-Employed,158466,95213,596,0,Good,4.308858,10.079462,Personal Loan
1,2,218,0.877,2019-06-15,21,Female,Secondary,West,Self-Employed,139129,67720,306,1,Average,4.735687,10.034368,Personal Loan
2,3,416,0.187,2019-01-14,21,Male,Graduate,South,Self-Employed,26663,40113,373,1,Average,7.576694,9.180393,Personal Loan
3,4,247,0.153,2023-05-09,42,Female,Post-Graduate,East,Unemployed,155955,94946,699,1,Poor,6.386953,8.390404,Home Loan
4,5,306,0.110,2020-03-10,41,Female,Graduate,South,Self-Employed,59359,16431,857,1,Good,6.697592,11.000303,Business Loan


# PART C: DATA UNDERSTANDING & CLEANING

### Data Exploration

Before cleaning the dataset, it is important to understand its structure.

We check:
- Number of rows and columns
- Data types
- Missing values
- Statistical summary

Why?
Because understanding data helps us choose the right preprocessing methods.

In [100]:
# Display shape of dataset
print("Shape of Dataset:")
print(df.shape)

# WHY?
# Shows number of rows and columns

Shape of Dataset:
(1000, 17)


In [101]:
# Display column names
print("Column Names:")
print(df.columns)

# WHY?
# Helps identify available features

Column Names:
Index(['customer_id', 'transaction_count', 'spending_ratio', 'join_date',
       'age', 'gender', 'education_level', 'region', 'employment_type',
       'annual_income', 'loan_amount', 'credit_score', 'default_flag',
       'repayment_history', 'inflation_rate', 'interest_rate', 'loan_purpose'],
      dtype='object')


In [102]:
# Display information about dataset
print("Dataset Information:")
df.info()

# WHY?
# Shows:
# - Data types
# - Null values
# - Memory usage

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        1000 non-null   int64  
 1   transaction_count  1000 non-null   int64  
 2   spending_ratio     1000 non-null   float64
 3   join_date          1000 non-null   object 
 4   age                1000 non-null   int64  
 5   gender             1000 non-null   object 
 6   education_level    1000 non-null   object 
 7   region             1000 non-null   object 
 8   employment_type    1000 non-null   object 
 9   annual_income      1000 non-null   int64  
 10  loan_amount        1000 non-null   int64  
 11  credit_score       1000 non-null   int64  
 12  default_flag       1000 non-null   int64  
 13  repayment_history  1000 non-null   object 
 14  inflation_rate     1000 non-null   float64
 15  interest_rate      1000 non-null   float64
 16  loan

In [103]:
# Statistical summary
print("Statistical Summary:")
display(df.describe())

# WHY?
# Gives:
# - Mean
# - Median
# - Min / Max
# - Standard deviation

Statistical Summary:


,customer_id,transaction_count,spending_ratio,age,annual_income,loan_amount,credit_score,default_flag,inflation_rate,interest_rate
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,255.989000,0.494061,45.237000,109128.636000,51796.066000,596.277000,0.485000,6.098191,9.076106
std,288.819436,142.208052,0.286756,14.391454,51770.846474,26751.921222,174.398237,0.500025,1.180108,1.736683
min,1.000000,10.000000,0.001000,21.000000,20039.000000,5028.000000,300.000000,0.000000,4.008670,6.000671
25%,250.750000,135.000000,0.248500,32.000000,63991.250000,29425.000000,450.500000,0.000000,5.040455,7.541417
50%,500.500000,258.000000,0.488000,45.500000,111942.500000,51465.500000,584.500000,0.000000,6.155127,9.084033
75%,750.250000,374.250000,0.738000,58.000000,154158.750000,74290.250000,747.000000,1.000000,7.155214,10.616445
max,1000.000000,499.000000,0.998000,69.000000,199925.000000,99919.000000,899.000000,1.000000,7.998830,11.999282


In [104]:
# Missing values count
print("Missing Values:")
display(df.isnull().sum())

# WHY?
# Identifies which columns need cleaning

Missing Values:


customer_id          0
transaction_count    0
spending_ratio       0
join_date            0
age                  0
gender               0
education_level      0
region               0
employment_type      0
annual_income        0
loan_amount          0
credit_score         0
default_flag         0
repayment_history    0
inflation_rate       0
interest_rate        0
loan_purpose         0
dtype: int64

## Handling Missing Values

### 1. SIMPLE IMPUTER (NUMERICAL: MEAN / MEDIAN)

#### SimpleImputer (Numerical)

- Mean → replaces missing values with average
- Median → replaces missing values with middle value

Usage:
- Mean → normal distribution
- Median → skewed data / outliers

In [105]:
from sklearn.impute import SimpleImputer

# Median for age
median_imputer = SimpleImputer(strategy="median")
df["age"] = median_imputer.fit_transform(df[["age"]])

# Mean for income and credit score
mean_imputer = SimpleImputer(strategy="mean")
df["annual_income"] = mean_imputer.fit_transform(df[["annual_income"]])
df["credit_score"] = mean_imputer.fit_transform(df[["credit_score"]])

print("After Numerical Imputation:")
display(df[["age", "annual_income", "credit_score"]].head())

After Numerical Imputation:


,age,annual_income,credit_score
0,54.0,158466.0,596.0
1,21.0,139129.0,306.0
2,21.0,26663.0,373.0
3,42.0,155955.0,699.0
4,41.0,59359.0,857.0


### 2. SIMPLE IMPUTER (CATEGORICAL: MOST FREQUENT)
### SimpleImputer (Categorical)

Most frequent value (mode) is used for categorical variables.

Why?
- Preserves most common category
- Simple and effective

In [106]:
from sklearn.impute import SimpleImputer

cat_imputer = SimpleImputer(strategy="most_frequent")

# FIX: use .ravel()
df["employment_type"] = cat_imputer.fit_transform(
    df[["employment_type"]]
).ravel()

print("After Categorical Imputation:")
display(df[["employment_type"]].head())

After Categorical Imputation:


,employment_type
0,Self-Employed
1,Self-Employed
2,Self-Employed
3,Unemployed
4,Self-Employed


### 3. MOST FREQUENT CATEGORY IMPUTATION (MANUAL)

#### Most Frequent Category (Manual)

Mode can also be applied manually using pandas.

In [107]:
df["employment_type"] = df["employment_type"].fillna(
    df["employment_type"].mode()[0]
)

### 4. COMPLETE CASE ANALYSIS (DROP ROWS / COLUMNS)
### Complete Case Analysis

Rows or columns with missing values are removed.

Why?
- Simple method
- Used when missing values are very few

In [108]:
# Drop rows with missing values
df_drop_rows = df.dropna()

# Drop columns with missing values
df_drop_cols = df.dropna(axis=1)

print("Shape after dropping rows:", df_drop_rows.shape)
print("Shape after dropping columns:", df_drop_cols.shape)

Shape after dropping rows: (1000, 17)
Shape after dropping columns: (1000, 17)


### 5. MISSING INDICATOR + RANDOM SAMPLING

### Missing Indicator + Random Sampling

Steps:
1. Create indicator column
2. Fill missing values using random samples

Why?
- Preserves data distribution
- Captures missingness information

In [109]:
random_vals = pd.Series(
    df["annual_income"].dropna().sample(len(df), replace=True).values,
    index=df.index
)

df["annual_income"] = df["annual_income"].fillna(random_vals)

### 6. KNN IMPUTER (MULTIVARIATE)

### KNN Imputer

KNN uses similarity between rows:

Steps:
1. Find nearest neighbors
2. Compute average
3. Fill missing values

Why?
- Uses relationships between features
- More accurate

In [110]:
from sklearn.impute import KNNImputer

knn = KNNImputer(n_neighbors=5)

df[["annual_income", "loan_amount", "credit_score"]] = knn.fit_transform(
    df[["annual_income", "loan_amount", "credit_score"]]
)

print("After KNN Imputation:")
display(df[["annual_income", "credit_score"]].head())

After KNN Imputation:


,annual_income,credit_score
0,158466.0,596.0
1,139129.0,306.0
2,26663.0,373.0
3,155955.0,699.0
4,59359.0,857.0


### 7. MICE ALGORITHM (ADVANCED)

### MICE (Multiple Imputation)

MICE fills missing values iteratively:

- Predicts missing values using other features
- Repeats multiple times

Why?
- More accurate
- Handles complex relationships

In [111]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice = IterativeImputer(max_iter=5, random_state=42)

df[["annual_income", "loan_amount", "credit_score"]] = mice.fit_transform(
    df[["annual_income", "loan_amount", "credit_score"]]
)

print("After MICE Imputation:")
display(df[["annual_income", "credit_score"]].head())

After MICE Imputation:


,annual_income,credit_score
0,158466.0,596.0
1,139129.0,306.0
2,26663.0,373.0
3,155955.0,699.0
4,59359.0,857.0


In [112]:
print("Final Missing Values:")
display(df.isnull().sum())

Final Missing Values:


customer_id          0
transaction_count    0
spending_ratio       0
join_date            0
age                  0
gender               0
education_level      0
region               0
employment_type      0
annual_income        0
loan_amount          0
credit_score         0
default_flag         0
repayment_history    0
inflation_rate       0
interest_rate        0
loan_purpose         0
dtype: int64

# PART D: OUTLIER DETECTION & HANDLING

### Step 1: Introduction

#### Outlier Detection

Outliers are extreme values that are significantly different from other observations.

They can:
- Distort statistical results
- Affect model performance
- Reduce accuracy

Hence, detecting and handling outliers is an important preprocessing step.

### Step 2: Select Numerical Columns

#### Selecting Numerical Features

Outliers are detected only in numerical columns:

- age
- annual_income
- loan_amount
- credit_score
- transaction_count

In [113]:
# List of numerical columns

num_cols = [
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "transaction_count"
]

print("Numerical Columns:")
print(num_cols)

Numerical Columns:
['age', 'annual_income', 'loan_amount', 'credit_score', 'transaction_count']


## METHOD 1: Z-SCORE METHOD

### Step 3: Formula

#### Z-Score Method

Z-score measures how far a value is from the mean.

If |Z| > 3 → Outlier

Z = \frac{X - \mu}{\sigma}

### Step 4: Apply Z-Score

In [114]:
from scipy.stats import zscore
import numpy as np

# Calculate Z-score
z_scores = np.abs(zscore(df[num_cols]))

# Count outliers
print("Z-score Outliers Count:")
print((z_scores > 3).sum())

Z-score Outliers Count:
0


### Step 5: Remove Outliers

#### Removing Z-Score Outliers

Rows where Z-score > 3 are removed.

In [115]:
df_z = df[(z_scores < 3).all(axis=1)]

print("Shape after removing Z-score outliers:")
print(df_z.shape)

Shape after removing Z-score outliers:
(1000, 17)


## METHOD 2: IQR METHOD

### Step 6: Formula
#### IQR Method

IQR = Q3 - Q1

Outliers:
- Below Q1 - 1.5 × IQR
- Above Q3 + 1.5 × IQR


### Step 7: Detect Outliers

In [116]:
# Detect outliers using IQR

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col} → Outliers:", outliers.shape[0])

age → Outliers: 0
annual_income → Outliers: 0
loan_amount → Outliers: 0
credit_score → Outliers: 0
transaction_count → Outliers: 0


### Step 8: Remove IQR Outliers

In [117]:
df_iqr = df.copy()

for col in num_cols:
    Q1 = df_iqr[col].quantile(0.25)
    Q3 = df_iqr[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_iqr = df_iqr[
        (df_iqr[col] >= lower) &
        (df_iqr[col] <= upper)
    ]

print("Shape after IQR filtering:")
print(df_iqr.shape)

Shape after IQR filtering:
(1000, 17)


## METHOD 3: CAPPING (WINSORIZATION)

### Step 9: Explanation

#### Capping Method

Instead of removing outliers, we limit extreme values.

Why?
- Prevents data loss
- Keeps dataset size unchanged

### Step 10: Apply Capping

In [118]:
df_cap = df.copy()

for col in num_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)

    df_cap[col] = np.where(df_cap[col] < lower, lower, df_cap[col])
    df_cap[col] = np.where(df_cap[col] > upper, upper, df_cap[col])

print("After Capping:")
display(df_cap[num_cols].head())

After Capping:


,age,annual_income,loan_amount,credit_score,transaction_count
0,54.0,158466.0,95213.0,596.0,348.0
1,21.0,139129.0,67720.0,306.0,218.0
2,21.0,26663.0,40113.0,373.0,416.0
3,42.0,155955.0,94946.0,699.0,247.0
4,41.0,59359.0,16431.0,857.0,306.0


## METHOD:4 PERCENTILE METHOD (OUTLIER HANDLING)

### Step 11: Explanation

#### Percentile Method

Percentile method is used to detect outliers by defining upper and lower limits.

Common approach:
- Lower limit → 1st percentile (1%)
- Upper limit → 99th percentile (99%)

Values outside this range are considered outliers.

Why use?
- Works well for large datasets
- Less sensitive than Z-score
- Easy to implement

### Step 12: Apply Percentile Method

In [119]:
# Apply percentile method

df_percentile = df.copy()

for col in num_cols:
    
    # Calculate percentiles
    lower = df[col].quantile(0.01)   # 1st percentile
    upper = df[col].quantile(0.99)   # 99th percentile

    # Detect outliers
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    
    print(f"{col} → Outliers:", outliers.shape[0])

age → Outliers: 0
annual_income → Outliers: 20
loan_amount → Outliers: 20
credit_score → Outliers: 15
transaction_count → Outliers: 18


### Step 13: Capping Using Percentile (Better Method)

#### Percentile Capping

Instead of removing data, extreme values are replaced with percentile limits.

Why?
- Prevents data loss
- Keeps dataset size same

In [120]:
# Apply percentile capping

df_cap_percentile = df.copy()

for col in num_cols:
    
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)

    df_cap_percentile[col] = np.where(
        df_cap_percentile[col] < lower,
        lower,
        df_cap_percentile[col]
    )

    df_cap_percentile[col] = np.where(
        df_cap_percentile[col] > upper,
        upper,
        df_cap_percentile[col]
    )

print("After Percentile Capping:")
display(df_cap_percentile[num_cols].head())

After Percentile Capping:


,age,annual_income,loan_amount,credit_score,transaction_count
0,54.0,158466.0,95213.0,596.0,348.0
1,21.0,139129.0,67720.0,306.0,218.0
2,21.0,26663.0,40113.0,373.0,416.0
3,42.0,155955.0,94946.0,699.0,247.0
4,41.0,59359.0,16431.0,857.0,306.0


### Final Summary of Outlier Handling Techniques

In this project, multiple methods were used to detect and handle outliers:

1. Z-Score Method:
   - Measures deviation from mean
   - Outliers identified when |Z| > 3
   - Suitable for normally distributed data

2. IQR (Interquartile Range) Method:
   - Based on quartiles (Q1, Q3)
   - Outliers lie outside:
     Q1 - 1.5 × IQR and Q3 + 1.5 × IQR
   - Works well for skewed data

3. Percentile Method:
   - Uses lower (1%) and upper (99%) thresholds
   - Values outside this range are treated as outliers
   - Effective for large datasets

4. Capping (Winsorization):
   - Replaces extreme values instead of removing them
   - Prevents data loss
   - Maintains dataset size

Overall Benefits:
- Improves data quality
- Reduces noise
- Enhances model performance

# PART E: FEATURE ENGINEERING

## Step 1: Introduction

### Feature Engineering

Feature Engineering is the process of transforming raw data into meaningful features.

It helps:
- Improve model performance
- Capture hidden patterns
- Convert data into machine-readable format

### 1. HANDLE VARIABLE TYPES (MIXED VARIABLES)

#### Handling Mixed Variables

Dataset contains both:
- Numerical variables (age, income)
- Categorical variables (gender, region)

These are handled separately for proper processing.

In [121]:
# Check data types

print("Data Types:")
print(df.dtypes)

# WHY?
# Helps identify which columns are numerical and categorical

Data Types:
customer_id            int64
transaction_count      int64
spending_ratio       float64
join_date             object
age                  float64
gender                object
education_level       object
region                object
employment_type       object
annual_income        float64
loan_amount          float64
credit_score         float64
default_flag           int64
repayment_history     object
inflation_rate       float64
interest_rate        float64
loan_purpose          object
dtype: object


### 2. DATE & TIME VARIABLES

### Date Feature Extraction

The join_date column is converted into datetime format.

From this, we extract:
- Year
- Month
- Day
- Weekday

This helps capture time-based patterns.

In [122]:
# Convert to datetime
df["join_date"] = pd.to_datetime(df["join_date"])

# Extract features
df["year"] = df["join_date"].dt.year
df["month"] = df["join_date"].dt.month
df["day"] = df["join_date"].dt.day
df["weekday"] = df["join_date"].dt.weekday

print("Date Features:")
display(df[["join_date", "year", "month", "day", "weekday"]].head())

Date Features:


,join_date,year,month,day,weekday
0,2019-01-23,2019,1,23,2
1,2019-06-15,2019,6,15,5
2,2019-01-14,2019,1,14,0
3,2023-05-09,2023,5,9,1
4,2020-03-10,2020,3,10,1


### 3. ENCODING CATEGORICAL VARIABLES

#### (A) ORDINAL ENCODING (education_level)

#### Ordinal Encoding

Ordinal variables have a meaningful order.

Example:
Primary < Secondary < Graduate < Post-Graduate

In [123]:
# Ordinal encoding

education_order = {
    "Primary": 1,
    "Secondary": 2,
    "Graduate": 3,
    "Post-Graduate": 4
}

df["education_level"] = df["education_level"].map(education_order)

print("Ordinal Encoding:")
display(df[["education_level"]].head())

Ordinal Encoding:


,education_level
0,1
1,2
2,3
3,4
4,3


#### (B) LABEL ENCODING (BINARY FEATURES)

#### Label Encoding

Binary variables are converted into 0 and 1.

Example:
Male → 1
Female → 0

In [124]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["gender"] = le.fit_transform(df["gender"])

print("Label Encoding:")
display(df[["gender"]].head())

Label Encoding:


,gender
0,0
1,0
2,1
3,0
4,0


#### (C) ONE HOT ENCODING

#### One Hot Encoding

Used for nominal categorical variables:

- region
- loan_purpose

Each category is converted into a separate column.

In [125]:
df = pd.get_dummies(
    df,
    columns=["region", "loan_purpose"],
    drop_first=True
)

print("One Hot Encoding:")
display(df.head())

One Hot Encoding:


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,employment_type,annual_income,loan_amount,...,year,month,day,weekday,region_North,region_South,region_West,loan_purpose_Education Loan,loan_purpose_Home Loan,loan_purpose_Personal Loan
0,1,348,0.246,2019-01-23,54.0,0,1,Self-Employed,158466.0,95213.0,...,2019,1,23,2,False,True,False,False,False,True
1,2,218,0.877,2019-06-15,21.0,0,2,Self-Employed,139129.0,67720.0,...,2019,6,15,5,False,False,True,False,False,True
2,3,416,0.187,2019-01-14,21.0,1,3,Self-Employed,26663.0,40113.0,...,2019,1,14,0,False,True,False,False,False,True
3,4,247,0.153,2023-05-09,42.0,0,4,Unemployed,155955.0,94946.0,...,2023,5,9,1,False,False,False,False,True,False
4,5,306,0.110,2020-03-10,41.0,0,3,Self-Employed,59359.0,16431.0,...,2020,3,10,1,False,True,False,False,False,False


### 4. ENCODING NUMERICAL FEATURES

#### (A) BINNING (INCOME GROUPS)

#### Binning

Continuous data is divided into categories.

Income groups:
- Low
- Medium
- High

In [126]:
df["income_group"] = pd.cut(
    df["annual_income"],
    bins=[0, 50000, 100000, 200000],
    labels=["Low", "Medium", "High"]
)

display(df[["annual_income", "income_group"]].head())

,annual_income,income_group
0,158466.0,High
1,139129.0,High
2,26663.0,Low
3,155955.0,High
4,59359.0,Medium


#### (B) BINARIZATION

#### Binarization

Converts numerical values into binary (0/1).

Example:
Income > 100000 → 1 (High)
Else → 0

In [127]:
df["high_income_flag"] = (df["annual_income"] > 100000).astype(int)

display(df[["annual_income", "high_income_flag"]].head())

,annual_income,high_income_flag
0,158466.0,1
1,139129.0,1
2,26663.0,0
3,155955.0,1
4,59359.0,0


#### (C) QUANTILE BINNING

#### Quantile Binning

Divides data into equal-sized groups (quartiles).

Why?
- Ensures balanced distribution

In [128]:
df["income_quantile"] = pd.qcut(
    df["annual_income"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

display(df[["annual_income", "income_quantile"]].head())

,annual_income,income_quantile
0,158466.0,Q4
1,139129.0,Q3
2,26663.0,Q1
3,155955.0,Q4
4,59359.0,Q1


#### (D) K-MEANS BINNING

#### K-Means Binning

Uses clustering to group similar values.

Why?
- Data-driven grouping
- Better than manual bins

In [129]:
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings("ignore")

# Reshape data
kmeans = KMeans(n_clusters=3, random_state=42)

df["income_kmeans"] = kmeans.fit_predict(
    df[["annual_income"]]
)

display(df[["annual_income", "income_kmeans"]].head())

,annual_income,income_kmeans
0,158466.0,1
1,139129.0,0
2,26663.0,2
3,155955.0,1
4,59359.0,2


### Feature Engineering Summary

- Handled mixed variable types
- Extracted date features (year, month, day, weekday)
- Applied encoding:
  - Ordinal encoding
  - Label encoding
  - One-hot encoding
- Transformed numerical features:
  - Binning
  - Binarization
  - Quantile binning
  - K-means binning

These techniques improve data representation and model performance.

# PART F: Applying Multiple Scaling Methods

## Objective

Machine Learning models me different features (income, loan amount, credit score, etc.) alag scale me hote hain.
Isliye hum feature scaling use karte hain taaki model biased na ho aur better performance de.

In [130]:
df = df.copy()

# Convert date into numeric feature
df["join_date"] = pd.to_datetime(df["join_date"])
df["join_year"] = df["join_date"].dt.year

# Select ONLY numerical features for scaling
features = [
    "transaction_count",
    "spending_ratio",
    "age",
    "annual_income",
    "loan_amount",
    "credit_score",
    "inflation_rate",
    "interest_rate",
    "join_year"
]

## STANDARDIZATION (Z-SCORE SCALING)

Explanation:

Standardization data ko mean = 0 aur standard deviation = 1 me convert karta hai.
Ye tab use hota hai jab data normal distribution ya ML models (Logistic Regression, SVM) ke liye prepare karna ho.

z = \frac{x - \mu}{\sigma}

In [131]:
from sklearn.preprocessing import StandardScaler

# Standardization
# Use: When data has different units and we need normalized distribution

scaler = StandardScaler()

df_standard = df.copy()
df_standard[features] = scaler.fit_transform(df[features])

print("STANDARDIZED DATA (Z-score scaling):")
display(df_standard.head())

STANDARDIZED DATA (Z-score scaling):


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,employment_type,annual_income,loan_amount,...,region_South,region_West,loan_purpose_Education Loan,loan_purpose_Home Loan,loan_purpose_Personal Loan,income_group,high_income_flag,income_quantile,income_kmeans,join_year
0,1,0.647341,-0.865492,2019-01-23,0.609208,0,1,Self-Employed,0.953472,1.623758,...,True,False,False,False,True,High,1,Q4,1,-0.808090
1,2,-0.267270,1.336085,2019-06-15,-1.684967,0,2,Self-Employed,0.579774,0.595542,...,False,True,False,False,True,High,1,Q3,0,-0.808090
2,3,1.125752,-1.071345,2019-01-14,-1.684967,1,3,Self-Employed,-1.593694,-0.436937,...,True,False,False,False,True,Low,0,Q1,2,-0.808090
3,4,-0.063242,-1.189972,2023-05-09,-0.225038,0,4,Unemployed,0.904946,1.613773,...,False,False,False,True,False,High,1,Q4,1,1.741090
4,5,0.351851,-1.340000,2020-03-10,-0.294558,0,3,Self-Employed,-0.961826,-1.322625,...,True,False,False,False,False,Medium,0,Q1,2,-0.170795


## NORMALIZATION

Explanation:

Normalization data ko 0 se 1 range me convert karta hai.
- Use hota hai Neural Networks aur distance-based models ke liye.

In [132]:
from sklearn.preprocessing import MinMaxScaler

# Normalization
# Use: When we need bounded values between 0 and 1

minmax = MinMaxScaler()

df_norm = df.copy()
df_norm[features] = minmax.fit_transform(df[features])

print("NORMALIZED DATA (0 to 1 range):")
display(df_norm.head())

NORMALIZED DATA (0 to 1 range):


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,employment_type,annual_income,loan_amount,...,region_South,region_West,loan_purpose_Education Loan,loan_purpose_Home Loan,loan_purpose_Personal Loan,income_group,high_income_flag,income_quantile,income_kmeans,join_year
0,1,0.691207,0.245737,2019-01-23,0.687500,0,1,Self-Employed,0.769526,0.950406,...,True,False,False,False,True,High,1,Q4,1,0.2
1,2,0.425358,0.878636,2019-06-15,0.000000,0,2,Self-Employed,0.662030,0.660674,...,False,True,False,False,True,High,1,Q3,0,0.2
2,3,0.830266,0.186560,2019-01-14,0.000000,1,3,Self-Employed,0.036823,0.369740,...,True,False,False,False,True,Low,0,Q1,2,0.2
3,4,0.484663,0.152457,2023-05-09,0.437500,0,4,Unemployed,0.755567,0.947593,...,False,False,False,True,False,High,1,Q4,1,1.0
4,5,0.605317,0.109328,2020-03-10,0.416667,0,3,Self-Employed,0.218583,0.120169,...,True,False,False,False,False,Medium,0,Q1,2,0.4


## MIN-MAX SCALING

Explanation:

Min-Max Scaling bhi normalization jaisa hota hai, sab values ko fixed range (0–1) me map karta hai.
- Use: Feature comparison aur neural networks

In [133]:
# Min-Max Scaling
# Use: When all features need same scale

minmax_scaler = MinMaxScaler()

df_minmax = df.copy()
df_minmax[features] = minmax_scaler.fit_transform(df[features])

print("MIN-MAX SCALED DATA:")
display(df_minmax.head())

MIN-MAX SCALED DATA:


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,employment_type,annual_income,loan_amount,...,region_South,region_West,loan_purpose_Education Loan,loan_purpose_Home Loan,loan_purpose_Personal Loan,income_group,high_income_flag,income_quantile,income_kmeans,join_year
0,1,0.691207,0.245737,2019-01-23,0.687500,0,1,Self-Employed,0.769526,0.950406,...,True,False,False,False,True,High,1,Q4,1,0.2
1,2,0.425358,0.878636,2019-06-15,0.000000,0,2,Self-Employed,0.662030,0.660674,...,False,True,False,False,True,High,1,Q3,0,0.2
2,3,0.830266,0.186560,2019-01-14,0.000000,1,3,Self-Employed,0.036823,0.369740,...,True,False,False,False,True,Low,0,Q1,2,0.2
3,4,0.484663,0.152457,2023-05-09,0.437500,0,4,Unemployed,0.755567,0.947593,...,False,False,False,True,False,High,1,Q4,1,1.0
4,5,0.605317,0.109328,2020-03-10,0.416667,0,3,Self-Employed,0.218583,0.120169,...,True,False,False,False,False,Medium,0,Q1,2,0.4


## MAX-ABS SCALING
 Explanation:

Max-Abs Scaling data ko -1 se 1 range me convert karta hai without shifting mean.
- Use: Sparse data / already centered data

In [134]:
from sklearn.preprocessing import MaxAbsScaler

# Max-Abs Scaling
# Use: Sparse data or when data already centered

maxabs = MaxAbsScaler()

df_maxabs = df.copy()
df_maxabs[features] = maxabs.fit_transform(df[features])

print("MAX-ABS SCALED DATA:")
display(df_maxabs.head())

MAX-ABS SCALED DATA:


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,employment_type,annual_income,loan_amount,...,region_South,region_West,loan_purpose_Education Loan,loan_purpose_Home Loan,loan_purpose_Personal Loan,income_group,high_income_flag,income_quantile,income_kmeans,join_year
0,1,0.697395,0.246493,2019-01-23,0.782609,0,1,Self-Employed,0.792627,0.952902,...,True,False,False,False,True,High,1,Q4,1,0.998023
1,2,0.436874,0.878758,2019-06-15,0.304348,0,2,Self-Employed,0.695906,0.677749,...,False,True,False,False,True,High,1,Q3,0,0.998023
2,3,0.833667,0.187375,2019-01-14,0.304348,1,3,Self-Employed,0.133365,0.401455,...,True,False,False,False,True,Low,0,Q1,2,0.998023
3,4,0.494990,0.153307,2023-05-09,0.608696,0,4,Unemployed,0.780068,0.950230,...,False,False,False,True,False,High,1,Q4,1,1.000000
4,5,0.613226,0.110220,2020-03-10,0.594203,0,3,Self-Employed,0.296906,0.164443,...,True,False,False,False,False,Medium,0,Q1,2,0.998517


## ROBUST SCALING
 Explanation:

Robust Scaling median aur IQR use karta hai.
- Ye outliers ko ignore karke stable scaling deta hai.

- Best for: Credit scoring / financial datasets 

In [135]:
from sklearn.preprocessing import RobustScaler

# Robust Scaling
# Use: When dataset contains outliers (income, loan_amount etc.)

robust = RobustScaler()

df_robust = df.copy()
df_robust[features] = robust.fit_transform(df[features])

print("ROBUST SCALED DATA:")
display(df_robust.head())

ROBUST SCALED DATA:


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,employment_type,annual_income,loan_amount,...,region_South,region_West,loan_purpose_Education Loan,loan_purpose_Home Loan,loan_purpose_Personal Loan,income_group,high_income_flag,income_quantile,income_kmeans,join_year
0,1,0.376176,-0.494382,2019-01-23,0.326923,0,1,Self-Employed,0.515968,0.975087,...,True,False,False,False,True,High,1,Q4,1,-0.333333
1,2,-0.167189,0.794688,2019-06-15,-0.942308,0,2,Self-Employed,0.301511,0.362296,...,False,True,False,False,True,High,1,Q3,0,-0.333333
2,3,0.660397,-0.614913,2019-01-14,-0.942308,1,3,Self-Employed,-0.945790,-0.253035,...,True,False,False,False,True,Low,0,Q1,2,-0.333333
3,4,-0.045977,-0.684372,2023-05-09,-0.134615,0,4,Unemployed,0.488119,0.969135,...,False,False,False,True,False,High,1,Q4,1,1.000000
4,5,0.200627,-0.772217,2020-03-10,-0.173077,0,3,Self-Employed,-0.583176,-0.780883,...,True,False,False,False,False,Medium,0,Q1,2,0.000000


## FINAL SUMMARY: SCALING METHODS

### Why Scaling is Used
Scaling is used in Machine Learning to bring all numerical features to a similar range so that no feature dominates the model. It improves accuracy and performance.

---

### 1️ STANDARDIZATION (Z-SCORE)
- Converts data to mean = 0, std = 1  
- Best for SVM, Logistic Regression, PCA  
- Good for normally distributed data  

---

### 2️ NORMALIZATION
- Scales data between 0 and 1  
- Best for Neural Networks and KNN  
- Sensitive to outliers  

---

### 3️ MIN-MAX SCALING
- Same as normalization (0 to 1 range)  
- Keeps original distribution shape  
- Simple but affected by outliers  

---

### 4️ MAX-ABS SCALING
- Scales data between -1 and 1  
- Works well with sparse data  
- Does not shift mean  

---

### 5️ ROBUST SCALING
- Uses median and IQR  
- Best for data with outliers   
- Ideal for financial/credit datasets  

---

##  FINAL CONCLUSION
For credit dataset, **Robust Scaling is best** because it handles outliers and gives stable results.

# PART G: FEATURE CONSTRUCTION & TRANSFORMATION

## FUNCTION TRANSFORMER

##  Explanation
FunctionTransformer is used to apply mathematical transformations on features.  
It helps in reducing skewness and improving data distribution.

Common transformations:
- Log transformation → reduces right skewness  
- Square root transformation → reduces moderate skewness  
- Reciprocal transformation → handles highly skewed data  

## Log Transformation

In [136]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

# Log transform (reduces right skewness in income/loan)
log_tf = FunctionTransformer(np.log1p)

df_log = df.copy()
df_log["annual_income"] = log_tf.fit_transform(df[["annual_income"]])

print("Log Transformed Income:")
display(df_log[["annual_income"]].head())

Log Transformed Income:


,annual_income
0,11.973302
1,11.843164
2,10.191070
3,11.957329
4,10.991376


## Square Root Transformation

In [137]:
# Square root reduces moderate skewness
sqrt_tf = FunctionTransformer(np.sqrt)

df_sqrt = df.copy()
df_sqrt["loan_amount"] = sqrt_tf.fit_transform(df[["loan_amount"]])

print("Square Root Loan Amount:")
display(df_sqrt[["loan_amount"]].head())

Square Root Loan Amount:


,loan_amount
0,308.566038
1,260.230667
2,200.282301
3,308.133088
4,128.183462


## Reciprocal Transformation

In [138]:
# Reciprocal used for highly skewed data
reciprocal_tf = FunctionTransformer(lambda x: 1/(x+1))

df_rec = df.copy()
df_rec["transaction_count"] = reciprocal_tf.fit_transform(df[["transaction_count"]])

print("Reciprocal Transaction Count:")
display(df_rec[["transaction_count"]].head())

Reciprocal Transaction Count:


,transaction_count
0,0.002865
1,0.004566
2,0.002398
3,0.004032
4,0.003257


## POWER TRANSFORMER

Explanation

PowerTransformer is used to make data more Gaussian-like (normal distribution).
It improves model performance by stabilizing variance and reducing skewness.

Types:

Box-Cox → only positive values
Yeo-Johnson → works with both positive and negative values

In [139]:
from sklearn.preprocessing import PowerTransformer

# Yeo-Johnson transformation
pt = PowerTransformer(method="yeo-johnson")

df_power = df.copy()

df_power[["annual_income", "loan_amount"]] = pt.fit_transform(
    df[["annual_income", "loan_amount"]]
)

print("Power Transformed Data:")
display(df_power[["annual_income", "loan_amount"]].head())

Power Transformed Data:


,annual_income,loan_amount
0,0.946791,1.508022
1,0.609238,0.629545
2,-1.709678,-0.357345
3,0.903538,1.499848
4,-0.936952,-1.369784


## COLUMN TRANSFORMER

Explanation

ColumnTransformer allows applying different preprocessing steps to different columns in a single pipeline.

Example:

Numerical columns → scaling
Categorical columns → encoding

This is widely used in real-world ML pipelines.

In [143]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# simple column transformer
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), ["age", "annual_income", "loan_amount"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["gender", "education_level"])
])

df_transformed = preprocessor.fit_transform(df)

print("Column Transformer Applied Successfully")

Column Transformer Applied Successfully


## FEATURE CONSTRUCTION

#### 1. Debt-to-Income Ratio
 Explanation

Measures financial risk by comparing loan amount to income.

In [144]:
df["debt_to_income"] = df["loan_amount"] / df["annual_income"]

print("Debt-to-Income Ratio:")
display(df[["debt_to_income"]].head())

Debt-to-Income Ratio:


,debt_to_income
0,0.600842
1,0.486743
2,1.504444
3,0.608804
4,0.276807


#### 2. Average Monthly Transactions
 Explanation

Shows customer activity level per month.

In [145]:
df["avg_monthly_transactions"] = df["transaction_count"] / 12

print("Average Monthly Transactions:")
display(df[["avg_monthly_transactions"]].head())

Average Monthly Transactions:


,avg_monthly_transactions
0,29.000000
1,18.166667
2,34.666667
3,20.583333
4,25.500000


#### 3. Spending-to-Income Ratio
 Explanation

Indicates spending behavior relative to income.

In [146]:
df["spending_to_income"] = df["spending_ratio"] * df["annual_income"]

print("Spending-to-Income Ratio:")
display(df[["spending_to_income"]].head())

Spending-to-Income Ratio:


,spending_to_income
0,38982.636
1,122016.133
2,4985.981
3,23861.115
4,6529.490


##  FINAL SUMMARY

###  Overall Concept
Feature engineering improves machine learning models by transforming raw data into more meaningful and useful features.  
It helps the model learn patterns better and improves overall performance.

---

### Techniques Used

- **FunctionTransformer** → Applies mathematical transformations like log, square root, and reciprocal to reduce skewness.  
- **PowerTransformer** → Transforms data to make it more normally distributed (Gaussian-like).  
- **ColumnTransformer** → Applies different preprocessing steps to different columns in one pipeline.  
- **Feature Construction** → Creates new meaningful business features from existing data (e.g., ratios, averages).

---

### FINAL CONCLUSION

Feature transformation and construction help to:

- Improve model accuracy  
- Reduce skewness and variance in data  
- Handle real-world financial datasets effectively  
- Create meaningful business insights for better decision-making  

# PART H: FINAL DELIVERABLE

In [149]:
# Final dataset (after all preprocessing steps)
final_df = df.copy()

print("Final Dataset Shape:")
print(final_df.shape)

display(final_df.head())

Final Dataset Shape:
(1000, 33)


,customer_id,transaction_count,spending_ratio,join_date,age,gender,education_level,employment_type,annual_income,loan_amount,...,loan_purpose_Home Loan,loan_purpose_Personal Loan,income_group,high_income_flag,income_quantile,income_kmeans,join_year,debt_to_income,avg_monthly_transactions,spending_to_income
0,1,348,0.246,2019-01-23,54.0,0,1,Self-Employed,158466.0,95213.0,...,False,True,High,1,Q4,1,2019,0.600842,29.000000,38982.636
1,2,218,0.877,2019-06-15,21.0,0,2,Self-Employed,139129.0,67720.0,...,False,True,High,1,Q3,0,2019,0.486743,18.166667,122016.133
2,3,416,0.187,2019-01-14,21.0,1,3,Self-Employed,26663.0,40113.0,...,False,True,Low,0,Q1,2,2019,1.504444,34.666667,4985.981
3,4,247,0.153,2023-05-09,42.0,0,4,Unemployed,155955.0,94946.0,...,True,False,High,1,Q4,1,2023,0.608804,20.583333,23861.115
4,5,306,0.110,2020-03-10,41.0,0,3,Self-Employed,59359.0,16431.0,...,False,False,Medium,0,Q1,2,2020,0.276807,25.500000,6529.490


In [153]:
# Final cleaned dataset
final_df = df.copy()

# Save as CSV file
final_df.to_csv(r"C:\Users\tanaa\Downloads\customer_credit_final_dataset.csv", index=False)

print("File saved successfully!")

File saved successfully!
